# Pandas Data Cleaning, Joins and Aggregation

**Type:** Original supplementary practice  
**Data:** Synthetic employee and training records

This notebook demonstrates common tabular data operations, including missing-value handling, duplicate detection, string normalisation, joins, grouping and pivot tables.

In [ ]:
import numpy as np
import pandas as pd

employees = pd.DataFrame({
    "employee_id":[101,102,103,104,105,105],
    "name":["Amina","  Rahim ","Sadia","Tanvir",None,None],
    "department":["IT","Finance","IT","Operations","Finance","Finance"],
    "salary":[72000,68000,np.nan,59000,63000,63000],
    "join_date":["2022-03-01","2021-08-15","2023-01-12","2020-11-04","2022-06-23","2022-06-23"]
})
training = pd.DataFrame({
    "employee_id":[101,102,103,106],
    "course":["Python","Excel","SQL","Power BI"],
    "score":[88,76,92,81]
})
display(employees)
display(training)

## 1. Inspect and clean

In [ ]:
print("Shape:", employees.shape)
print("Duplicate IDs:", employees["employee_id"].duplicated(keep=False).sum())
display(employees.isna().sum().to_frame("missing"))
clean = employees.drop_duplicates(subset="employee_id", keep="first").copy()
clean["name"] = clean["name"].str.strip().str.title()
clean["join_date"] = pd.to_datetime(clean["join_date"], errors="coerce")
clean["salary"] = clean["salary"].fillna(clean.groupby("department")["salary"].transform("median"))
display(clean)

## 2. Join datasets

An inner join retains employees with a training record. A left join would preserve every employee and show missing training information where no match exists.

In [ ]:
joined = clean.merge(training, on="employee_id", how="left", validate="one_to_one")
display(joined)
print("Unmatched employees:", joined["course"].isna().sum())

## 3. Group and pivot

In [ ]:
department_summary = clean.groupby("department", as_index=False).agg(
    employee_count=("employee_id","nunique"),
    mean_salary=("salary","mean"),
    median_salary=("salary","median")
)
display(department_summary.round(2))

pivot = pd.pivot_table(
    joined, index="department", columns="course",
    values="score", aggfunc="mean"
)
display(pivot)

## 4. Date features and resampling

In [ ]:
clean["join_month"] = clean["join_date"].dt.to_period("M").dt.to_timestamp()
monthly_hires = clean.groupby("join_month").size().rename("hires")
display(monthly_hires.to_frame())

## Practice prompts

- Compare median and mean salary by department. When do they differ?
- What is the difference between `drop_duplicates(keep="first")` and `keep="last"`?
- What assumptions are required for a one-to-one merge?
- Why may filling a missing salary with a department median be inappropriate in an actual HR dataset?

**Privacy note:** Never publish identifiable employee or customer data without a lawful basis and appropriate permission. The records here are fictional.